# Case-001 Lab Recipient Qualification Gate

Public-safe executable check for qualifying a possible lab recipient before any
approved preclinical quote packet is sent. This notebook is not contact
permission, treatment advice, trial screening, or a place for private records.


In [ ]:
recipient_classes = {
    "quote_admin_or_cro_feasibility": {
        "can_support_quote_lane": True,
        "required_outputs": [
            "cost",
            "timeline",
            "sample_requirements",
            "capability_route",
        ],
    },
    "erythroid_hbf_screening_lab": {
        "can_support_quote_lane": True,
        "required_outputs": ["HBG_or_HbF", "maturation", "viability", "controls"],
    },
    "primary_erythroid_or_disease_model_lab": {
        "can_support_quote_lane": True,
        "required_outputs": ["ethics_constraints", "disease_model", "raw_data"],
    },
    "mature_red_cell_safety_lab": {
        "can_support_quote_lane": True,
        "required_outputs": ["hemolysis", "red_cell_source", "rejection_threshold"],
    },
    "clinical_trial_or_therapy_center": {
        "can_support_quote_lane": False,
        "required_outputs": ["benchmark_only"],
    },
}

blocked_requests = {
    "raw_records",
    "patient_samples",
    "treatment_selection",
    "dosing",
    "trial_screening",
    "travel_import_purchase",
    "procurement_guidance",
}

## Deterministic Gate Check

A recipient can support the quote lane only if it is a preclinical-capability
recipient and does not trigger blocked requests.


In [ ]:
def classify_recipient(recipient_class: str, requested_items: set[str]) -> str:
    """Classify a possible recipient for the public-safe quote lane."""
    if requested_items & blocked_requests:
        return "reject_for_quote_lane"

    recipient = recipient_classes[recipient_class]
    if not recipient["can_support_quote_lane"]:
        return "benchmark_only_not_quote_lab"

    if not recipient["required_outputs"]:
        return "hold_missing_outputs"

    return "candidate_recipient_pass"


assert (
    classify_recipient("clinical_trial_or_therapy_center", set())
    == "benchmark_only_not_quote_lab"
)
assert (
    classify_recipient("erythroid_hbf_screening_lab", {"raw_records"})
    == "reject_for_quote_lane"
)
assert (
    classify_recipient("quote_admin_or_cro_feasibility", set())
    == "candidate_recipient_pass"
)

## Source-Checked Snapshot

Counts and identifiers come from official PubMed and ClinicalTrials.gov checks
run on 2026-06-08.


In [ ]:
source_snapshot = {
    "pubmed_assay_model_query_2024_2026_count": 246,
    "assay_anchor_pmids": [41259521, 39108322, 39504332, 36769243, 33879818, 15163316],
    "clinicaltrials_hbf_cd34_first_records": [
        "NCT03655678",
        "NCT06291961",
        "NCT06328764",
        "NCT06065189",
        "NCT03282656",
    ],
    "decision": "hold_until_founder_approval_and_recipient_pass",
}

summary = {
    "quote_capable_classes": sum(
        1 for item in recipient_classes.values() if item["can_support_quote_lane"]
    ),
    "benchmark_only_classes": sum(
        1 for item in recipient_classes.values() if not item["can_support_quote_lane"]
    ),
    "blocked_request_count": len(blocked_requests),
    "decision": source_snapshot["decision"],
}
summary